# Chapter 16.2. 신경망 RL 리뷰 — 리뷰어의 재실행

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter16_2_neural_review.ipynb)

책 본문: [Chapter 16.2](https://smhanlab.com/book-ml/kor/ml2/chapter16.html) (Peer-Review)

이 노트북은 본문 §"손으로 한 번"의 **가상 발표**(팀 "진자돌림": Pendulum-v1,
PPO, \(\gamma=0.99\), \(\lambda=0.95\), 300 반복 × 400 스텝 = 12만 스텝,
시드 42)를 **리뷰어의 눈으로 재실행**한다. 발표팀의 실험을 그대로 재현한 뒤,
각 "반박 가능한 질문"의 *답이 되는 실험*을 순서대로 돌려,
"PPO가 잘 작동했다"는 결론이 그 숫자들로 뒤집히는지 확인한다.

1. **발표팀 재현** — 시드 42, 12만 스텝 → 처음/마지막 10에피소드 평균
2. **패턴 1 검증** — 시드 123, 7 추가 → 시드 간 편차
3. **패턴 2 검증** — 24만 스텝으로 확장(포화 여부) + 학습 후 재평가 갭
4. **패턴 3 검증** — 보상 1/8 스케일링 + \(\gamma=0.95\) 민감성
5. **패턴 5 검증** — 무작위 정책 300 에피소드 베이스라인 실측


## 0. 환경 준비

PPO 코드는 Chapter 11.2의 실습 노트북과 동일하다.


In [1]:
import os, math
import numpy as np
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

# 한국어 라벨을 폰트 'Noto Sans CJK KR'으로 (없으면 기본 폰트로 넘어가도 출력은 됨)
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr:
    plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print(f"torch: {torch.__version__}, gymnasium: {gym.__version__}")
print(f"그림 저장 위치: {IMG}")


torch: 2.13.0+cpu, gymnasium: 1.3.0
그림 저장 위치: /home/smhan/book-ml/kor/src/images


In [2]:
class ActorCritic(nn.Module):
    """PPO actor-critic (Chapter 11.2 코드). 가우시안 정책 mu + std, 가치 헤드 V."""
    def __init__(self, state_dim, act_dim, act_high):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(state_dim, 64), nn.Tanh(),
                                   nn.Linear(64, 64), nn.Tanh())
        self.mu = nn.Linear(64, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim) - 0.5)  # exp() 후 약 0.61
        self.v = nn.Linear(64, 1)
        self.act_high = act_high
    def forward(self, x):
        h = self.trunk(x)
        mu = torch.tanh(self.mu(h)) * self.act_high  # 행동 범위에 맞게 스케일
        std = torch.exp(self.log_std)
        return mu, std, self.v(h).squeeze(-1)

def compute_gae(rewards, values, last_value, gamma=0.99, lam=0.95):
    """GAE (Ch11.3): A_t = sum_k (gamma*lam)^k * delta_{t+k}"""
    T = len(rewards)
    adv = [0.0] * T
    last_gae = 0.0
    for t in reversed(range(T)):
        next_v = values[t + 1] if t < T - 1 else last_value
        delta = rewards[t] + gamma * next_v - values[t]
        last_gae = delta + gamma * lam * last_gae
        adv[t] = last_gae
    rets = [a + v for a, v in zip(adv, values)]
    return adv, rets

def train_ppo(seed, n_iter, steps=400, gamma=0.99, lam=0.95,
              reward_scale=1.0, eps=0.2, n_epoch=4, mb=64, verbose=False):
    """PPO(Ch11.2 프로토콜)로 학습. (에피소드별 리턴, 모델) 반환."""
    torch.manual_seed(seed); np.random.seed(seed)
    env = gym.make("Pendulum-v1")
    ac = ActorCritic(env.observation_space.shape[0], env.action_space.shape[0],
                     env.action_space.high[0])
    opt = torch.optim.Adam(ac.parameters(), lr=3e-4)

    episode_returns = []
    cursor = 0
    for it in range(n_iter):
        state, _ = env.reset(seed=seed + it)
        states, actions, logps, rewards, values = [], [], [], [], []
        for _ in range(steps):
            st = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                mu, std, v = ac(st)
            dist = torch.distributions.Normal(mu, std)
            a = dist.sample()
            states.append(state); actions.append(a)
            logps.append(dist.log_prob(a).sum())
            next_state, r, term, trunc, _ = env.step(a.squeeze(0).numpy())
            rewards.append(r * reward_scale); values.append(v.item())
            if term or trunc:   # Pendulum은 truncation: 200 스텝만 지나도 종료
                episode_returns.append(float(np.sum(rewards[cursor:])))
                cursor = len(rewards)
                state, _ = env.reset()
            else:
                state = next_state
        states_t = torch.tensor(np.asarray(states), dtype=torch.float32)
        actions_t = torch.tensor(np.asarray(actions), dtype=torch.float32)
        logps_t = torch.stack(logps)
        rewards_t = torch.tensor(rewards, dtype=torch.float32)
        values_t = torch.tensor(values, dtype=torch.float32)
        with torch.no_grad():
            last_v = ac(states_t[-1:])[2].item()
        adv, rets = compute_gae(rewards_t.tolist(), values_t.tolist(), last_v, gamma, lam)
        adv_t = torch.tensor(adv, dtype=torch.float32)
        ret_t = torch.tensor(rets, dtype=torch.float32)
        adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)  # 어드밴티지 정규화
        idx = torch.randperm(len(rewards))
        for _ in range(n_epoch):          # 같은 데이터를 4에폭 재사용 (이후 버림)
            for start in range(0, len(rewards), mb):
                b = idx[start:start + mb]
                mu, std, v = ac(states_t[b])
                dist = torch.distributions.Normal(mu, std)
                new_logps = dist.log_prob(actions_t[b]).sum(-1)
                ratio = torch.exp(new_logps - logps_t[b])       # r_t(theta)
                surr1 = ratio * adv_t[b]
                surr2 = torch.clamp(ratio, 1 - eps, 1 + eps) * adv_t[b]
                pol_loss = -torch.min(surr1, surr2).mean()
                val_loss = F.mse_loss(v, ret_t[b])
                entropy = dist.entropy().sum(-1).mean()
                loss = pol_loss + 0.5 * val_loss - 0.01 * entropy
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(ac.parameters(), 0.5)
                opt.step()
        if verbose and (it % 50 == 0 or it == n_iter - 1):
            print(f"iter {it:3d}: last10={np.mean(episode_returns[-10:]):.1f}")
    env.close()
    return episode_returns, ac

def evaluate(ac, episodes=20, deterministic=False):
    """학습 *후* 재평가: 에피소드 리턴 리스트.
    deterministic=True면 탐험 없이 mu만 쓰는 결정론적 평가."""
    env = gym.make("Pendulum-v1")
    rets = []
    for e in range(episodes):
        state, _ = env.reset(seed=777000 + e)
        total, done = 0.0, False
        while not done:
            st = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                mu, std, _ = ac(st)
            if deterministic:
                a = mu
            else:
                a = torch.distributions.Normal(mu, std).sample()
            state, r, term, trunc, _ = env.step(a.squeeze(0).numpy())
            total += r
            done = term or trunc
        rets.append(total)
    env.close()
    return rets


## 1. 발표팀 재현 (시드 42, 12만 스텝)

발표팀의 실험을 그대로 실행한다: 300 반복 × 400 스텝,
\(\gamma=0.99\), \(\lambda=0.95\), 클리핑 \(\epsilon=0.2\),
보상 스케일링 없음, 시드 42.


In [3]:
N_ITER, STEPS = 300, 400   # 총 12만 스텝 (발표팀 프로토콜)
GAMMA, LAM = 0.99, 0.95

returns_42, ac42 = train_ppo(42, N_ITER, STEPS, GAMMA, LAM, verbose=True)

first10 = np.mean(returns_42[:10])
last10  = np.mean(returns_42[-10:])
print(f"총 에피소드: {len(returns_42)}")
print(f"처음 10 에피소드 평균 리턴:  {first10:.1f}")
print(f"마지막 10 에피소드 평균 리턴: {last10:.1f}")
print(f"개선폭(마지막10 - 처음10): {last10 - first10:.1f}")


iter   0: last10=-1320.7


iter  50: last10=-665.6


iter 100: last10=-655.7


iter 150: last10=-728.5


iter 200: last10=-604.9


iter 250: last10=-724.0


iter 299: last10=-741.4
총 에피소드: 600
처음 10 에피소드 평균 리턴:  -805.3
마지막 10 에피소드 평균 리턴: -741.4
개선폭(마지막10 - 처음10): 63.9


In [4]:
ep = np.arange(1, len(returns_42) + 1)
sm = np.convolve(returns_42, np.ones(25) / 25, mode="valid")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ep, returns_42, color="lightgray", lw=0.7, alpha=0.8)
ax.plot(ep[24:], sm, color="tab:blue", lw=1.8)
ax.axhline(first10, color="gray", ls="--", lw=1)
ax.text(len(returns_42) * 0.01, first10 + 80,
        f"first 10 episodes mean {first10:.0f}", fontsize=8, color="dimgray")
ax.axhline(last10, color="tab:red", ls="--", lw=1)
ax.text(len(returns_42) * 0.55, last10 + 80,
        f"last 10 episodes mean {last10:.0f}", fontsize=8, color="tab:red")
ax.set_xlabel("Episode number"); ax.set_ylabel("Return (Pendulum, lower is better)")
ax.set_title("Presenter reproduction — PPO, Pendulum-v1 (seed 42, 120k steps)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch16_2_ppo_seed42_curve.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch16_2_ppo_seed42_curve.svg")


저장: /home/smhan/book-ml/kor/src/images/ch16_2_ppo_seed42_curve.svg


## 2. 패턴 1 검증 — 시드 2개를 더 돌리면?

"곡선이 완만하게 올라가네"는 *시드 42 하나로* 본 곡선이다.
같은 프로토콜로 시드 123, 7을 추가해서, **시드 간 편차**가
발표팀이 주장한 개선폭보다 큰지 본다(Ch11.2의 초기화 민감성).


In [5]:
seed_runs = {42: returns_42}
for s in [123, 7]:
    r, _ = train_ppo(s, N_ITER, STEPS, GAMMA, LAM, verbose=True)
    seed_runs[s] = r

hdr = "%5s %10s %10s %10s" % ("seed", "처음10", "마지막10", "개선폭")
print(hdr)
last10s, improvements = [], []
for s, r in seed_runs.items():
    f, l = np.mean(r[:10]), np.mean(r[-10:])
    last10s.append(l); improvements.append(l - f)
    print("%5d %10.1f %10.1f %10.1f" % (s, f, l, l - f))
last10s = np.array(last10s)
print(f"\n마지막10의 시드 간 표준편차: {last10s.std(ddof=1):.1f}")
print(f"시드별 개선폭: {[f'{x:.1f}' for x in improvements]}")


iter   0: last10=-1036.8


iter  50: last10=-702.2


iter 100: last10=-734.1


iter 150: last10=-750.9


iter 200: last10=-692.0


...

iter 100: last10=-745.2


iter 150: last10=-692.6


iter 200: last10=-706.7


iter 250: last10=-735.0


iter 299: last10=-573.0
 seed       처음10      마지막10        개선폭
   42     -805.3     -741.4       63.9
  123     -719.5     -658.0       61.4
    7     -636.9     -573.0       64.0

마지막10의 시드 간 표준편차: 84.2
시드별 개선폭: ['63.9', '61.4', '64.0']


## 3. 패턴 2 검증 — 24만 스텝과 재평가 갭

두 가지 실험: (a) 같은 시드 42로 **24만 스텝**까지 늘려, 12만 스텝에서
이동평균이 포화였는지 확인(11.2의 "절반도 안 왔다").
(b) 발표팀의 12만 스텝 모델을 학습 *후*에 따로 재평가 —
탐험(\(\sigma\)) 포함 vs \(\mu\)만 쓰는 결정론적 평가.


In [6]:
returns_42_240k, _ = train_ppo(42, 2 * N_ITER, STEPS, GAMMA, LAM, verbose=True)
mid10 = np.mean(returns_42_240k[:600][-10:])   # 12만 스텝 시점 (600 에피소드)
end10 = np.mean(returns_42_240k[-10:])         # 24만 스텝 시점
print(f"12만 스텝 시점 마지막10: {mid10:.1f}")
print(f"24만 스텝 시점 마지막10: {end10:.1f}")
print(f"추가 12만 스텝의 개선: {end10 - mid10:+.1f}")


iter   0: last10=-1320.7


iter  50: last10=-665.6


iter 100: last10=-655.7


iter 150: last10=-728.5


iter 200: last10=-604.9


...

iter 400: last10=-723.6


iter 450: last10=-685.1


iter 500: last10=-699.0


iter 550: last10=-611.8


iter 599: last10=-672.3
12만 스텝 시점 마지막10: -741.4
24만 스텝 시점 마지막10: -672.3
추가 12만 스텝의 개선: +69.2


In [7]:
# (b) 학습 도중 리턴 vs 학습 후 재평가 — 같은 정책, 다른 프로토콜
eval_explore = evaluate(ac42, episodes=20, deterministic=False)
eval_det = evaluate(ac42, episodes=20, deterministic=True)
print(f"학습 도중 마지막10 (발표팀이 보고한 값): {last10:.1f}")
print(f"재평가 — 탐험 포함(σ 샘플링) 20 에피소드: {np.mean(eval_explore):.1f}")
print(f"재평가 — 결정론적(μ, 탐험 0) 20 에피소드: {np.mean(eval_det):.1f}")


학습 도중 마지막10 (발표팀이 보고한 값): -741.4
재평가 — 탐험 포함(σ 샘플링) 20 에피소드: -1327.7
재평가 — 결정론적(μ, 탐험 0) 20 에피소드: -1319.2


## 4. 패턴 3 검증 — 보상 스케일링 민감성

보상을 **1/8로 스케일링**하고 \(\gamma=0.95\)로 재실행(16.1의 예시).
스케일링은 리턴의 *규모*를 직접 바꾸므로, "결과가 좋아진 것"이
알고리즘의 공헌인지 하이퍼파라미터의 공헌인지 분리해보기 어렵다.
(비교를 위해 그림에서 모든 리턴을 1/8 스케일로 환산해 겹쳐 그린다.)


In [8]:
returns_scaled, _ = train_ppo(42, N_ITER, STEPS, gamma=0.95, lam=0.95,
                              reward_scale=1.0 / 8.0, verbose=True)
f_s, l_s = np.mean(returns_scaled[:10]), np.mean(returns_scaled[-10:])
print(f"[스케일링 없음, γ=0.99]  처음10={first10:.1f}  마지막10={last10:.1f}")
print(f"[보상 ×1/8, γ=0.95]      처음10={f_s:.2f}  마지막10={l_s:.2f}")
print(f"  (×8 환산: {f_s*8:.1f} / {l_s*8:.1f})")


iter   0: last10=-165.1


iter  50: last10=-84.4


iter 100: last10=-103.6


iter 150: last10=-109.8


iter 200: last10=-99.1


iter 250: last10=-98.1


iter 299: last10=-94.0
[스케일링 없음, γ=0.99]  처음10=-805.3  마지막10=-741.4
[보상 ×1/8, γ=0.95]      처음10=-100.81  마지막10=-94.04
  (×8 환산: -806.5 / -752.3)


In [9]:
ep_s = np.arange(1, len(returns_scaled) + 1)
sm_u = np.convolve(returns_42, np.ones(25) / 25, mode="valid")
sm_s = np.convolve(returns_scaled, np.ones(25) / 25, mode="valid")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ep[24:], sm_u / 8.0, color="tab:blue", lw=1.8, label="no scaling (return/8)")
ax.plot(ep_s[24:], sm_s, color="tab:red", lw=1.8, label="reward x1/8, γ=0.95")
ax.set_xlabel("Episode number"); ax.set_ylabel("Return (÷8 scale)")
ax.set_title("Effect of reward scaling on learning curves (seed 42)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch16_2_reward_scale_curve.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch16_2_reward_scale_curve.svg")


저장: /home/smhan/book-ml/kor/src/images/ch16_2_reward_scale_curve.svg


## 5. 패턴 5 검증 — 무작위 정책 베이스라인

"잘 배웠다"의 기준점. 무작위 정책 300 에피소드의 평균 리턴을 실측한다.


In [10]:
env = gym.make("Pendulum-v1")
rand_rets = []
for e in range(300):
    state, _ = env.reset(seed=5000 + e)
    total, done = 0.0, False
    while not done:
        state, r, term, trunc, _ = env.step(env.action_space.sample())
        total += r
        done = term or trunc
    rand_rets.append(total)
env.close()
rand_rets = np.array(rand_rets)
print(f"무작위 정책 300 에피소드 평균 리턴: {rand_rets.mean():.1f}")
print(f"표준편차: {rand_rets.std(ddof=1):.1f}")


무작위 정책 300 에피소드 평균 리턴: -1224.5
표준편차: 276.8


## 6. 리뷰 요약 — 재실행 숫자 모음


In [11]:
print("=" * 66)
print("리뷰어 요약 — 재실행 숫자 (Pendulum 리턴, 높을수록 좋음)")
print("=" * 66)
rows = [
    ("발표팀 (시드42, 12만) 처음10 / 마지막10", f"{first10:.1f} / {last10:.1f}"),
    ("마지막10의 시드 간 표준편차 (42/123/7)", f"{last10s.std(ddof=1):.1f}"),
    ("24만 스텝 마지막10 (포화 확인)", f"{end10:.1f}   [12만: {mid10:.1f}]"),
    ("재평가: 탐험 포함 / 결정론적", f"{np.mean(eval_explore):.1f} / {np.mean(eval_det):.1f}"),
    ("보상 ×1/8 + γ=0.95 마지막10 (×8 환산)", f"{l_s * 8:.1f}"),
    ("무작위 정책 베이스라인 (300 에피소드)", f"{rand_rets.mean():.1f}"),
]
for k, v in rows:
    print(f"{k:<38} {v}")
print("=" * 66)


리뷰어 요약 — 재실행 숫자 (Pendulum 리턴, 높을수록 좋음)
발표팀 (시드42, 12만) 처음10 / 마지막10           -805.3 / -741.4
마지막10의 시드 간 표준편차 (42/123/7)            84.2
24만 스텝 마지막10 (포화 확인)                   -672.3   [12만: -741.4]
재평가: 탐험 포함 / 결정론적                      -1327.7 / -1319.2
보상 ×1/8 + γ=0.95 마지막10 (×8 환산)         -752.3
무작위 정책 베이스라인 (300 에피소드)                -1224.5


## 7. 리뷰에 무엇을 적는가

재실행 숫자는 발표팀의 숫자를 "교정"하기 위한 것이 아니라,
**결론**("PPO가 잘 작동했다")이 그 숫자들에 의해 뒤집히는지
확인하기 위한 것이다:

- 시드 간 표준편차가 개선폭보다 크면 → "개선"의 방향성조차 시드의 우연과
  구별 불가 (패턴 1)
- 결정론적 재평가가 무작위 정책보다 낮으면 → 12만 스텝에서 "잘 배웠다"는
  지지되지 않음 (패턴 2 + 5)
- 보상 스케일링이 결과의 방향을 바꾸면 → 결론이 알고리즘의 공헌이 아니라
  하이퍼파라미터의 공헌일 수 있음 (패턴 3)

반대로, 재실행이 발표팀의 결론을 *지지*한다면 — "시드 2개를 추가해
재실행했고, 이동평균 편차(±…)가 알고리즘 간 차이보다 작았습니다" —
그것 또한 3점 리뷰다. 반박 가능한 질문은 **답이 실험인** 질문이지,
답이 발표팀을 불리하게 나온다는 보장이 붙은 질문이 아니다.
